In [5]:
import pandas as pd
import numpy as np

In [6]:
quali_2025 = pd.read_csv('race_quali_2025.csv')
quali_2024 = pd.read_csv('race_quali_2024.csv')
quali_2023 = pd.read_csv('race_quali_2023.csv')
quali_2022 = pd.read_csv('race_quali_2022.csv')

In [103]:
quali_2022.to_csv("race_quali_2022.csv", index=False)
quali_2023.to_csv("race_quali_2023.csv", index=False)
quali_2024.to_csv("race_quali_2024.csv", index=False)
quali_2025.to_csv("race_quali_2025.csv", index=False)

In [44]:
kaggle_sprint_res = pd.read_csv('filler-datasets/kaggle_sprint_res.csv')
kaggle_sprint_res.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId
0,1,1061,830,9,33,2,1,1,1,3,17,25:38.426,1538426,14,1:30.013,1
1,2,1061,1,131,44,1,2,2,2,2,17,+1.430,1539856,17,1:29.937,1
2,3,1061,822,131,77,3,3,3,3,1,17,+7.502,1545928,17,1:29.958,1
3,4,1061,844,6,16,4,4,4,4,0,17,+11.278,1549704,16,1:30.163,1
4,5,1061,846,1,4,6,5,5,5,0,17,+24.111,1562537,16,1:30.566,1


In [57]:
races_kaggle = pd.read_csv('filler-datasets/races_filtered.csv')
kaggle_races = races_kaggle[races_kaggle['year']>2021]
kaggle_races.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68 entries, 60 to 148
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   raceId     68 non-null     int64 
 1   year       68 non-null     int64 
 2   round      68 non-null     int64 
 3   circuitId  68 non-null     int64 
 4   name       68 non-null     object
dtypes: int64(4), object(1)
memory usage: 3.2+ KB


In [71]:
len(quali_2024['Track'].value_counts())+len(quali_2023['Track'].value_counts())+len(quali_2022['Track'].value_counts())

68

In [73]:
sprint_res_kaggle = pd.read_csv('filler-datasets/kaggle_sprint_res.csv')
len(sprint_res_kaggle['raceId'].value_counts())

18

In [108]:
len(sprint_quali_2024['Track'].value_counts())+len(sprint_quali_2023['Track'].value_counts())+len(sprint_quali_2021['Track'].value_counts())

18

In [11]:
sprint_quali_2021 = pd.read_csv('sprint_quali_2021.csv')
sprint_quali_2023 = pd.read_csv('sprint_quali_2023.csv')
sprint_quali_2024 = pd.read_csv('sprint_quali_2024.csv')

In [28]:
def normalize_name_series(s):
    return (
        s.str.lower()
         .str.strip()
         .str.normalize('NFKD')
         .str.encode('ascii', errors='ignore')
         .str.decode('utf-8')
    )

def add_kaggle_driver_id(df, kaggle_driver):
    df = df.copy()

    df['driver_norm'] = normalize_name_series(df['Driver'])

    kd = kaggle_driver.copy()
    kd['driver_norm'] = normalize_name_series(
        kd['forename'] + ' ' + kd['surname']
    )

    df = df.merge(
        kd[['driverId', 'driver_norm']],
        on='driver_norm',
        how='left'
    )

    df.rename(columns={'driverId': 'kaggle_driver_id'}, inplace=True)
    df.drop(columns=['driver_norm'], inplace=True)

    return df


In [38]:
quali_2022 = add_kaggle_driver_id(quali_2022, kaggle_driver)
#quali_2023 = add_kaggle_driver_id(quali_2023, kaggle_driver)
#quali_2024 = add_kaggle_driver_id(quali_2024, kaggle_driver)
#quali_2025 = add_kaggle_driver_id(quali_2025, kaggle_driver)

In [107]:
sprint_quali_2021.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,kaggle_driver_id
0,China,1,44,Lewis Hamilton,Ferrari,1:31.212,1:31.484,1:30.849,15,1
1,China,2,1,Max Verstappen,Red Bull Racing Honda RBPT,1:31.916,1:31.521,1:30.867,12,830
2,China,3,81,Oscar Piastri,McLaren Mercedes,1:31.723,1:31.362,1:30.929,13,857
3,China,4,16,Charles Leclerc,Ferrari,1:31.518,1:31.561,1:31.057,15,844
4,China,5,63,George Russell,Mercedes,1:31.952,1:31.346,1:31.169,18,847


In [80]:
quali_2022['year'] = 2022
quali_2023['year'] = 2023
quali_2024['year'] = 2024

In [92]:
race_id_sequence = kaggle_races['raceId'].tolist()
def assign_race_ids_across_seasons(dfs, race_id_sequence):
    race_ptr = 0
    total_races = len(race_id_sequence)

    updated_dfs = []

    for df in dfs:
        df = df.copy()

        race_id_col = []
        tracks = df['Track'].values
        n = len(df)

        start = 0

        for i in range(1, n):
            if tracks[i] != tracks[i - 1]:
                if race_ptr >= total_races:
                    raise IndexError("Exceeded race_id_sequence length")

                race_id_col.extend(
                    [race_id_sequence[race_ptr]] * (i - start)
                )
                race_ptr += 1
                start = i

        # last block of this dataset
        if race_ptr >= total_races:
            raise IndexError("Exceeded race_id_sequence length")

        race_id_col.extend(
            [race_id_sequence[race_ptr]] * (n - start)
        )
        race_ptr += 1

        if len(race_id_col) != n:
            raise ValueError("raceId assignment length mismatch")

        df['raceId'] = race_id_col
        updated_dfs.append(df)

    # Final boundary checks
    if race_ptr != total_races:
        raise ValueError(
            f"RaceId mismatch: used {race_ptr} ids but Kaggle has {total_races}"
        )

    return updated_dfs

In [93]:
quali_2022, quali_2023, quali_2024 = assign_race_ids_across_seasons(
    [quali_2022, quali_2023, quali_2024],
    race_id_sequence
)

In [96]:
quali_2024.head()

,Track,Position,No,Driver,Team,Q1,Q2,Q3,Laps,kaggle_driver_id,year,raceId
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1:30.031,1:29.374,1:29.179,17,830,2024,1121
1,Bahrain,2,16,Charles Leclerc,Ferrari,1:30.243,1:29.165,1:29.407,19,844,2024,1121
2,Bahrain,3,63,George Russell,Mercedes,1:30.350,1:29.922,1:29.485,17,847,2024,1121
3,Bahrain,4,55,Carlos Sainz,Ferrari,1:29.909,1:29.573,1:29.507,18,832,2024,1121
4,Bahrain,5,11,Sergio Perez,Red Bull Racing Honda RBPT,1:30.221,1:29.932,1:29.537,17,815,2024,1121


In [98]:
def quali_time_to_sec(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()
    parts = val.split(':')

    try:
        if len(parts) == 2:
            # mm:ss.xxx
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 3:
            # hh:mm:ss.xxx (rare but exists)
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
    except:
        return np.nan

    return np.nan
def process_quali(df):
    df = df.copy()

    # participation flags
    df['participated_q2'] = df['Q2'].notna().astype(int)
    df['participated_q3'] = df['Q3'].notna().astype(int)

    # parse times
    df['q1_time_sec'] = df['Q1'].apply(quali_time_to_sec)
    df['q2_time_sec'] = df['Q2'].apply(quali_time_to_sec)
    df['q3_time_sec'] = df['Q3'].apply(quali_time_to_sec)

    # keep final columns
    df = df[
        [
            'raceId',
            'Track',
            'Team',
            'kaggle_driver_id',
            'Position',
            'participated_q2',
            'participated_q3',
            'q1_time_sec',
            'q2_time_sec',
            'q3_time_sec',
        ]
    ]

    return df


In [70]:
import numpy as np

cols = ['SQ1', 'SQ2', 'SQ3']

existing_cols = sprint_quali_2021.columns.intersection(cols)

sprint_quali_2021[existing_cols] = (
    sprint_quali_2021[existing_cols]
    .replace('DNF', np.nan)
)

In [63]:
def sprint_quali_time_to_sec(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()
    parts = val.split(':')

    try:
        if len(parts) == 2:
            # mm:ss.xxx
            m, s = parts
            return int(m) * 60 + float(s)
        elif len(parts) == 3:
            # hh:mm:ss.xxx (rare but exists)
            h, m, s = parts
            return int(h) * 3600 + int(m) * 60 + float(s)
    except:
        return np.nan

    return np.nan
def sprint_process_quali(df):
    df = df.copy()

    # participation flags
    df['participated_q2'] = df['SQ2'].notna().astype(int)
    df['participated_q3'] = df['SQ3'].notna().astype(int)

    # parse times
    df['q1_time_sec'] = df['SQ1'].apply(sprint_quali_time_to_sec)
    df['q2_time_sec'] = df['SQ2'].apply(sprint_quali_time_to_sec)
    df['q3_time_sec'] = df['SQ3'].apply(sprint_quali_time_to_sec)

    # keep final columns
    df = df[
        [
            'raceId',
            'Track',
            'Team',
            'kaggle_driver_id',
            'Position',
            'participated_q2',
            'participated_q3',
            'q1_time_sec',
            'q2_time_sec',
            'q3_time_sec',
        ]
    ]

    return df


In [101]:
#quali_2022 = process_quali(quali_2022)
quali_2023 = process_quali(quali_2023)
quali_2024 = process_quali(quali_2024)